# 06 · Window length, data efficiency and explainability — RQ3

> **RQ3.** How do the input window length, forecast horizon and amount of training data affect DLinear and NLinear,
> and how efficient and interpretable are they compared with tree-based models?

| | |
|---|---|
| **Inputs** | `processed/*`, `results/rq1_fits.csv`, `results/rq2_fits_full_range.csv`, hypotheses from 04 and 05 |
| **Outputs** | `results/rq3_*.csv`, `results/rq3_linear_weights.npz`, `results/all_hypotheses.csv`, `figures/06_*.png` |

### Pre-registered hypotheses

* **H3a** — longer input windows reduce DLinear's error more than NLinear's (DLinear's trend branch needs context;
  NLinear is anchored on the last value).
* **H3b** — reducing the number of training cells degrades NLinear less than DLinear.
* **H3c** — DLinear and NLinear use at least 10× fewer parameters than the tree ensembles have nodes, and train at least
  10× faster. *Descriptive: checked against the stated thresholds, no significance test.*

H3a and H3b form one Holm family. The tree models carried into RQ3 (`config.RQ3_TREE_MODELS`) give context in every
figure but are not part of the hypotheses.

### A note on comparing window lengths fairly

An origin at 25% of life needs at least *L* observed cycles, so longer windows lose early origins. Comparing errors
across *L* on different units would confound window length with forecast difficulty. Every comparison across *L*
below uses only the **common units** — (cell, origin) pairs that every window length in the grid can serve.

In [ ]:
# Make the repository importable, locally or on Google Colab.
import os
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    # Once per session:
    #   !git clone https://github.com/<your-account>/battery-rul-linear.git /content/battery-rul-linear
    #   %pip install -q -r /content/battery-rul-linear/requirements.txt
    # Raw data on Drive: MyDrive/battery_rul/data/NASA_PCoE/*.mat and .../CALCE_CX2/CX2_*/
    from google.colab import drive
    drive.mount("/content/drive")
    os.chdir("/content/battery-rul-linear")
    os.environ.setdefault("BATTERY_RUL_DATA", "/content/drive/MyDrive/battery_rul/data")

ROOT = Path.cwd().resolve()
if not (ROOT / "battery_rul").is_dir():
    ROOT = ROOT.parent                          # launched from notebooks/
os.environ.setdefault("BATTERY_RUL_ROOT", str(ROOT))
sys.path.insert(0, str(ROOT))

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from battery_rul import config
from battery_rul.dataset import cells_by_dataset, load_cells, loco_folds
from battery_rul.evaluation import UNIT_COLS, seed_average
from battery_rul.experiment import ExperimentSettings, baseline_spec, linear_spec, prepare_fold, run_experiment, tree_spec
from battery_rul.explain import (last_value_weight, recency_profile, recent_weight_share, tree_permutation_importance,
                                 tree_shap_importance, weight_row_sums)
from battery_rul.features import make_windows
from battery_rul.models.linear import dlinear_branch_share, effective_map, train_linear
from battery_rul.models.trees import train_tree
from battery_rul.plotting import (INK_2, SEQUENTIAL, color_of, metric_vs_setting, savefig, setup_style,
                                  weight_heatmap)
from battery_rul.report import HYPOTHESES, HypothesisResult, describe, finalize
from battery_rul.stats import spearman, wilcoxon
from battery_rul.utils import load_table, markdown_table, record_run, save_json, save_table, timer

pd.set_option("display.width", 170)
setup_style()
record_run("06_window_length_explainable")
cells = load_cells()
REF = config.LABELS.primary_reference
folds = loco_folds(cells, REF)
METRIC = config.PRIMARY_METRIC
DATASETS = list(cells_by_dataset(cells))
LINEARS = ["DLinear", "NLinear"]
rq3_specs = ([linear_spec(m) for m in LINEARS] + [baseline_spec("LinearTrend")] +
             [tree_spec(m, mode) for m, mode in config.RQ3_TREE_MODELS])
RQ3_LABELS = [s.label for s in rq3_specs]
print("models:", RQ3_LABELS)
print("window grid:", config.RQ3_INPUT_LENS, "| horizons:", config.RQ3_HORIZONS, "| seeds:", config.RQ3_SEEDS)

## 1. Window length × horizon grid

In [ ]:
grid_records = []
with timer("RQ3 window grid"):
    for ds in DATASETS:
        ds_folds = [f for f in folds if f.dataset == ds]
        for L in config.RQ3_INPUT_LENS[ds]:
            for H in config.RQ3_HORIZONS[ds]:
                settings = ExperimentSettings(reference=REF, arm="window_grid").with_window(ds, L, H)
                rec, _, _ = run_experiment(cells, ds_folds, rq3_specs, config.RQ3_SEEDS, settings, log=lambda m: None)
                grid_records.append(rec)
                print(f"  {ds} L={L:3d} H={H:3d}: {rec['cell'].nunique() if len(rec) else 0} test cells, "
                      f"{len(rec)} forecasts")
grid = pd.concat(grid_records, ignore_index=True)
save_table(grid, "rq3_window_grid_records")
grid_units = seed_average(grid, ["model", "input_len", "horizon", *UNIT_COLS])

# common units: (cell, origin) pairs every L in the grid can serve, per dataset and horizon
present = grid_units.groupby(["dataset", "horizon", "cell", "frac"])["input_len"].nunique().reset_index()
n_L = {ds: len(config.RQ3_INPUT_LENS[ds]) for ds in DATASETS}
present = present[present.apply(lambda r: r["input_len"] == n_L[r["dataset"]], axis=1)]
grid_common = grid_units.merge(present[["dataset", "horizon", "cell", "frac"]], on=["dataset", "horizon", "cell", "frac"])
save_table(grid_units, "rq3_window_grid_units")
print(f"common units per dataset: {present.groupby('dataset').size().to_dict()}")

In [ ]:
default_h = {ds: config.WINDOWS[ds].horizon for ds in DATASETS}
fig, axes = plt.subplots(1, len(DATASETS), figsize=(6.5 * len(DATASETS), 4), squeeze=False)
curve_rows = []
for ax, ds in zip(axes[0], DATASETS):
    sub = grid_common[(grid_common["dataset"] == ds) & (grid_common["horizon"] == default_h[ds])]
    curve = sub.groupby(["model", "input_len"])[METRIC].agg(["median", lambda s: s.quantile(0.25),
                                                            lambda s: s.quantile(0.75)]).reset_index()
    curve.columns = ["model", "input_len", "median", "q25", "q75"]
    curve["dataset"] = ds
    curve_rows.append(curve)
    metric_vs_setting(curve, "input_len", "median", ax=ax, band=("q25", "q75"),
                      title=f"{ds}: error vs window length (H = {default_h[ds]}, common units)")
    ax.set_ylabel("median relative RUL error (band = IQR)")
fig.tight_layout()
savefig(fig, "06_error_vs_window_length")
plt.show()
save_table(pd.concat(curve_rows), "rq3_error_vs_window_length")

In [ ]:
for ds in DATASETS:
    fig, axes = plt.subplots(1, len(LINEARS), figsize=(5.5 * len(LINEARS), 3.4), squeeze=False)
    for ax, m in zip(axes[0], LINEARS):
        sub = grid_units[(grid_units["dataset"] == ds) & (grid_units["model"] == m)]
        heat = sub.pivot_table(index="horizon", columns="input_len", values=METRIC, aggfunc="median")
        im = ax.imshow(heat.to_numpy(), cmap=SEQUENTIAL, aspect="auto", origin="lower")
        ax.set_xticks(range(heat.shape[1]), heat.columns)
        ax.set_yticks(range(heat.shape[0]), heat.index)
        for (i, j), v in np.ndenumerate(heat.to_numpy()):
            ax.text(j, i, f"{v:.2f}", ha="center", va="center", fontsize=8,
                    color="white" if v > np.nanmedian(heat.to_numpy()) else "#0b0b0b")
        ax.set(title=f"{ds} · {m}", xlabel="input length L", ylabel="horizon H")
        ax.grid(False)
        fig.colorbar(im, ax=ax, shrink=0.8, label="median rel. error")
    fig.suptitle("Median relative RUL error over the (L, H) grid (all available units per cell)", fontsize=10,
                 color=INK_2)
    fig.tight_layout()
    savefig(fig, f"06_grid_heatmap_{ds.lower()}")
    plt.show()

### H3a — does window length matter more for DLinear?

Per unit (default horizon, common units): the change in relative error from the shortest to the longest window in
the grid; negative means longer windows help. H3a predicts DLinear's change is more negative than NLinear's. The
Spearman correlation between *L* and error is reported alongside as a description of the shape (it needs at least
three window lengths).

In [ ]:
rows = []
for ds in DATASETS:
    sub = grid_common[(grid_common["dataset"] == ds) & (grid_common["horizon"] == default_h[ds])]
    for (m, cell, frac), g in sub[sub["model"].isin(LINEARS)].groupby(["model", "cell", "frac"]):
        g = g.sort_values("input_len")
        rows.append({"model": m, "dataset": ds, "cell": cell, "frac": frac,
                     "change_shortest_to_longest": float(g[METRIC].iloc[-1] - g[METRIC].iloc[0]),
                     "rho_L_error": spearman(g["input_len"], g[METRIC])["rho"]})
window_effect = pd.DataFrame(rows)
save_table(window_effect, "rq3_window_effect")
change = window_effect.pivot_table(index=["dataset", "cell", "frac"], columns="model",
                                   values="change_shortest_to_longest").dropna()
if {"DLinear", "NLinear"} <= set(change.columns) and len(change) >= 2:
    _, p3a = wilcoxon(change["DLinear"], change["NLinear"])
    eff3a = float(np.median(change["DLinear"] - change["NLinear"]))
else:
    p3a, eff3a = np.nan, np.nan
h3a = HypothesisResult("H3a", HYPOTHESES["H3a"],
                       "Wilcoxon on per-unit error change (shortest -> longest L), DLinear vs NLinear",
                       len(change), eff3a, "median(change DLinear - change NLinear)", p3a,
                       bool(np.isfinite(eff3a) and eff3a < 0))
window_effect.groupby("model")[["change_shortest_to_longest", "rho_L_error"]].describe().round(3)

## 2. Training-set size

Each fold is re-run with only *k* of its training cells, drawn at random (a different draw per seed, the same draw for
every model). Folds where *k* would exceed the available training cells use all of them.

In [ ]:
size_records = []
with timer("RQ3 training-size curve"):
    for k in config.RQ3_TRAIN_CELLS:
        for s in config.RQ3_SEEDS:
            settings = ExperimentSettings(reference=REF, arm="train_size", max_train_cells=k, subset_seed=s)
            rec, _, _ = run_experiment(cells, folds, rq3_specs, (s,), settings, log=lambda m: None)
            rec["k_requested"] = k
            size_records.append(rec)
size = pd.concat(size_records, ignore_index=True)
size = size.drop_duplicates(subset=["model", "seed", "dataset", "cell", "frac", "n_train_cells"])
save_table(size, "rq3_train_size_records")
size_units = seed_average(size, ["model", "n_train_cells", *UNIT_COLS])
save_table(size_units, "rq3_train_size_units")

curve = size_units.groupby(["dataset", "model", "n_train_cells"])[METRIC].median().rename("median").reset_index()
fig, axes = plt.subplots(1, len(DATASETS), figsize=(6.5 * len(DATASETS), 4), squeeze=False)
for ax, ds in zip(axes[0], DATASETS):
    metric_vs_setting(curve[curve["dataset"] == ds], "n_train_cells", "median", ax=ax,
                      title=f"{ds}: error vs number of training cells")
    ax.set_ylabel("median relative RUL error")
fig.tight_layout()
savefig(fig, "06_error_vs_training_cells")
plt.show()

### H3b — which linear model is more data-efficient?

Per unit: degradation = error with the fewest training cells − error with all training cells. H3b predicts DLinear
degrades more than NLinear.

In [ ]:
deg_rows = []
for (ds, m, cell, frac), g in size_units[size_units["model"].isin(LINEARS)].groupby(["dataset", "model", "cell", "frac"]):
    if g["n_train_cells"].nunique() < 2:
        continue
    lo = g.loc[g["n_train_cells"].idxmin(), METRIC]
    hi = g.loc[g["n_train_cells"].idxmax(), METRIC]
    deg_rows.append({"dataset": ds, "model": m, "cell": cell, "frac": frac, "degradation": lo - hi})
size_deg = (pd.DataFrame(deg_rows).pivot_table(index=["dataset", "cell", "frac"], columns="model",
                                               values="degradation").dropna()
            if deg_rows else pd.DataFrame(columns=LINEARS))
save_table(size_deg.reset_index(), "rq3_train_size_degradation")
if {"DLinear", "NLinear"} <= set(size_deg.columns) and len(size_deg) >= 2:
    _, p3b = wilcoxon(size_deg["DLinear"], size_deg["NLinear"])
    eff3b = float(np.median(size_deg["DLinear"] - size_deg["NLinear"]))
else:
    p3b, eff3b = np.nan, np.nan
h3b = HypothesisResult("H3b", HYPOTHESES["H3b"], "Wilcoxon on paired degradation (fewest vs all cells)",
                       len(size_deg), eff3b, "median(DLinear - NLinear) degradation", p3b,
                       bool(np.isfinite(eff3b) and eff3b > 0))
size_deg.describe().round(3)

## 3. Efficiency

From the main runs of notebooks 04 and 05 (default windows, all seeds). Size is parameters for the linear models and
total tree nodes for the ensembles; times are measured on the same machine in the same run.

In [ ]:
fits = pd.concat([load_table("rq1_fits"), load_table("rq2_fits_full_range")], ignore_index=True)
eff = fits.groupby(["dataset", "model"]).agg(
    size=("n_params", "median"), tree_nodes=("n_tree_nodes", "median"),
    train_seconds=("train_seconds", "median"), infer_ms_per_window=("infer_seconds_per_window", "median"),
).reset_index()
eff["size"] = np.where(eff["size"] > 0, eff["size"], eff["tree_nodes"])
eff["infer_ms_per_window"] *= 1000
eff = eff.drop(columns="tree_nodes")
eff = eff[~eff["model"].isin(["Persistence", "LinearTrend"])]
save_table(eff, "rq3_efficiency")
print(markdown_table(eff, floatfmt=".4g"))

fig, axes = plt.subplots(1, 2, figsize=(13, 0.35 * eff["model"].nunique() + 2))
agg = eff.groupby("model")[["size", "train_seconds"]].median().sort_values("size")
for ax, col, label in ((axes[0], "size", "parameters / tree nodes (log)"),
                       (axes[1], "train_seconds", "training time per fit, s (log)")):
    ax.barh(agg.index, agg[col], color=[color_of(m) for m in agg.index], height=0.6)
    ax.set(xscale="log", xlabel=label)
    ax.grid(axis="y", visible=False)
axes[0].set_title("Model size")
axes[1].set_title("Training cost")
fig.tight_layout()
savefig(fig, "06_efficiency")
plt.show()

In [ ]:
lin = eff[eff["model"].isin(LINEARS)]
tre = eff[eff["model"].str.contains(r"\(") & ~eff["model"].str.startswith("DecisionTree")]   # ensembles only
size_ratio = float(tre["size"].min() / lin["size"].max()) if len(tre) and len(lin) else np.nan
time_ratio = float(tre["train_seconds"].min() / lin["train_seconds"].max()) if len(tre) and len(lin) else np.nan
h3c_met = bool(size_ratio >= 10 and time_ratio >= 10)
print(f"smallest tree ensemble / largest of DLinear, NLinear: size x{size_ratio:.1f}, "
      f"training time x{time_ratio:.1f} (per dataset and model, medians over folds and seeds)")
h3c = HypothesisResult("H3c", HYPOTHESES["H3c"], "descriptive thresholds (>= 10x size and >= 10x training time)",
                       len(fits), min(size_ratio, time_ratio), "min(size ratio, time ratio)", np.nan, h3c_met,
                       verdict="met (descriptive)" if h3c_met else "not met (descriptive)",
                       note=f"size ratio {size_ratio:.1f}, training-time ratio {time_ratio:.1f}")

## 4. Explainability — linear models

A trained linear model *is* its weight matrix: each forecast step *h* is a weighted sum of the *L* input values. Below,
DLinear, NLinear and Linear are retrained (seed 0, default windows) on every fold, and their effective matrices *W* are
averaged across folds per dataset. Rows are forecast steps, columns are input positions (right-most = last observed
cycle); the scale is diverging, so red and blue are opposite signs.

In [ ]:
weights, lin_rows = {}, []
for ds in DATASETS:
    w = config.WINDOWS[ds]
    Ws = {m: [] for m in ("DLinear", "NLinear", "Linear", "DLinear_trend", "DLinear_remainder")}
    for f in (f for f in folds if f.dataset == ds):
        fd = prepare_fold(cells, f, ExperimentSettings(reference=REF))
        if fd is None:
            continue
        test_X, _, _ = make_windows(cells[f.test].capacity, w.input_len, w.horizon)
        for m in ("DLinear", "NLinear", "Linear"):
            model = train_linear(m, fd.training, w.input_len, w.horizon, w.ma_kernel, seed=0)
            em = effective_map(model)
            Ws[m].append(em["W"])
            row = {"dataset": ds, "fold": f.fold_id, "model": m,
                   "recent_quarter_weight_share": recent_weight_share(em["W"]),
                   "mean_row_sum": float(weight_row_sums(em["W"]).mean()),
                   "last_value_weight_h1": float(last_value_weight(em["W"])[0]),
                   "last_value_weight_hH": float(last_value_weight(em["W"])[-1])}
            if m == "DLinear":
                Ws["DLinear_trend"].append(em["W_trend"])
                Ws["DLinear_remainder"].append(em["W_seasonal"])
                row.update(dlinear_branch_share(model, test_X) if len(test_X) else {})
            lin_rows.append(row)
    for k, v in Ws.items():
        if v:
            weights[f"{ds}_{k}"] = np.mean(v, axis=0)
linear_explain = pd.DataFrame(lin_rows)
np.savez(config.RESULTS_DIR / "rq3_linear_weights.npz", **weights)
save_table(linear_explain, "rq3_linear_explain")
linear_explain.groupby(["dataset", "model"]).median(numeric_only=True).round(3)

In [ ]:
for ds in DATASETS:
    keys = [k for k in (f"{ds}_DLinear", f"{ds}_NLinear", f"{ds}_Linear") if k in weights]
    fig, axes = plt.subplots(1, len(keys), figsize=(5.4 * len(keys), 3.4), squeeze=False)
    for ax, k in zip(axes[0], keys):
        weight_heatmap(weights[k], k.replace("_", " · ") + " (effective W)", ax=ax, fig=fig)
    fig.tight_layout()
    savefig(fig, f"06_effective_weights_{ds.lower()}")
    plt.show()
    if f"{ds}_DLinear_trend" in weights:
        fig, axes = plt.subplots(1, 2, figsize=(10.8, 3.4))
        weight_heatmap(weights[f"{ds}_DLinear_trend"], f"{ds} · DLinear trend branch", ax=axes[0], fig=fig)
        weight_heatmap(weights[f"{ds}_DLinear_remainder"], f"{ds} · DLinear remainder branch", ax=axes[1], fig=fig)
        fig.tight_layout()
        savefig(fig, f"06_dlinear_branches_{ds.lower()}")
        plt.show()

In [ ]:
fig, axes = plt.subplots(1, len(DATASETS), figsize=(6.5 * len(DATASETS), 3.6), squeeze=False)
for ax, ds in zip(axes[0], DATASETS):
    for m in ("DLinear", "NLinear", "Linear"):
        k = f"{ds}_{m}"
        if k in weights:
            prof = recency_profile(weights[k])
            ax.plot(prof["lag"], prof["share"], marker="o", ms=3, label=m, color=color_of(m),
                    ls="-." if m == "Linear" else "-")
    ax.set(xlabel="lag (cycles before the forecast origin)", ylabel="share of |weight|",
           title=f"{ds}: where the linear models look", xlim=(None, -0.5))
    ax.invert_xaxis()
    ax.legend(fontsize=8)
fig.tight_layout()
savefig(fig, "06_recency_profiles")
plt.show()

## 5. Explainability — tree models

Permutation importance of the window features on each fold's **test-cell** windows, averaged across folds; and, if the
`shap` package is installed, mean |SHAP| for the first and last forecast step on one fold per dataset. Features are
grouped into *level* and *shape* using the dictionary from notebook 03.

In [ ]:
dictionary = pd.read_csv(config.PROCESSED_DIR / "feature_dictionary.csv")
kind_of = {r["feature"]: ("level" if r["kind"].startswith("level") else "shape") for _, r in dictionary.iterrows()}
imp_rows, shap_rows = [], []
for ds in DATASETS:
    w = config.WINDOWS[ds]
    for i, f in enumerate(f for f in folds if f.dataset == ds):
        fd = prepare_fold(cells, f, ExperimentSettings(reference=REF))
        if fd is None:
            continue
        Xt, Yt, _ = make_windows(cells[f.test].capacity, w.input_len, w.horizon)
        for name, mode in config.RQ3_TREE_MODELS:
            model = train_tree(name, fd.training.X_all, fd.training.Y_all, mode, seed=0)
            imp = tree_permutation_importance(model, Xt, Yt, n_repeats=3 if config.QUICK else 5)
            imp["model"], imp["dataset"], imp["fold"] = f"{name} ({mode})", ds, f.fold_id
            imp_rows.append(imp)
            if i == 0:
                sh = tree_shap_importance(model, fd.training.X_all)
                if sh is not None:
                    sh["model"], sh["dataset"] = f"{name} ({mode})", ds
                    shap_rows.append(sh)
importance = pd.concat(imp_rows, ignore_index=True)
importance["kind"] = importance["feature"].map(kind_of).fillna("shape")
save_table(importance, "rq3_tree_permutation_importance")
if shap_rows:
    save_table(pd.concat(shap_rows, ignore_index=True), "rq3_tree_shap")
print("SHAP available:", bool(shap_rows))

imp_mean = importance.groupby(["dataset", "model", "feature", "kind"])["importance"].mean().reset_index()
fig, axes = plt.subplots(len(DATASETS), len(config.RQ3_TREE_MODELS),
                         figsize=(6 * len(config.RQ3_TREE_MODELS), 3.6 * len(DATASETS)), squeeze=False)
for r, ds in enumerate(DATASETS):
    for c, (name, mode) in enumerate(config.RQ3_TREE_MODELS):
        label = f"{name} ({mode})"
        top = imp_mean[(imp_mean["dataset"] == ds) & (imp_mean["model"] == label)].nlargest(10, "importance")[::-1]
        ax = axes[r][c]
        ax.barh(top["feature"], top["importance"], color=[color_of(label) if k == "level" else "#b5b3ab"
                                                          for k in top["kind"]], height=0.6)
        ax.set(title=f"{ds} · {label}", xlabel="permutation importance (MSE increase)")
        ax.grid(axis="y", visible=False)
fig.suptitle("Top-10 tree features (model colour = level feature, grey = shape feature)", fontsize=10,
             color=INK_2)
fig.tight_layout()
savefig(fig, "06_tree_importance")
plt.show()

level_share = (imp_mean.assign(pos=imp_mean["importance"].clip(lower=0))
               .groupby(["dataset", "model", "kind"])["pos"].sum().unstack(fill_value=0))
level_share["level_share"] = level_share.get("level", 0) / level_share.sum(axis=1).replace(0, np.nan)
save_table(level_share.reset_index(), "rq3_tree_level_share")
level_share[["level_share"]].round(3)

## 6. RQ3 verdicts and the study-wide hypothesis table

In [ ]:
rq3 = finalize([h3a, h3b]) + [h3c]
save_json([r.as_dict() for r in rq3], "rq3_hypotheses")
save_table(pd.DataFrame([r.as_dict() for r in rq3]), "rq3_hypotheses")
print(describe(rq3))

frames = []
for rq, name in (("RQ1", "rq1_hypotheses"), ("RQ2", "rq2_hypotheses"), ("RQ3", "rq3_hypotheses")):
    try:
        frames.append(load_table(name).assign(rq=rq))
    except FileNotFoundError:
        print(f"{name} missing - run the earlier notebook first")
all_h = pd.concat(frames, ignore_index=True)[["rq", "hypothesis", "n_units", "effect_label", "effect", "p_value",
                                               "p_holm", "verdict"]]
save_table(all_h, "all_hypotheses")
print("\n" + markdown_table(all_h, floatfmt=".4g"))

## 7. Summary

In [ ]:
for ds in DATASETS:
    sub = pd.concat(curve_rows)
    sub = sub[(sub["dataset"] == ds) & sub["model"].isin(LINEARS)]
    for m in LINEARS:
        s = sub[sub["model"] == m].sort_values("median")
        if len(s):
            print(f"{ds} {m}: best L = {int(s.iloc[0]['input_len'])} (median rel. error {s.iloc[0]['median']:.3f}), "
                  f"worst L = {int(s.iloc[-1]['input_len'])} ({s.iloc[-1]['median']:.3f})")
le = linear_explain.groupby(["model"])["recent_quarter_weight_share"].median()
print(f"\nShare of |weight| on the most recent quarter of the window: " +
      ", ".join(f"{m} {v:.0%}" for m, v in le.items()))
print(f"Tree level-feature share of importance: " +
      ", ".join(f"{i[1]} [{i[0]}] {v:.0%}" for i, v in level_share["level_share"].items()))
print()
print(describe(rq3))